In [1]:
# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

# Step 2: Upload CSV file (for Google Colab)
from google.colab import files
uploaded = files.upload()

# Load dataset
df = pd.read_csv('heart.csv')  # make sure file name matches
print(df.head())

Saving heart.csv to heart.csv
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [3]:
X = df.drop('HeartDisease', axis=1)   # change 'target' if your column name differs
y = df['HeartDisease']

In [4]:
# Identify categorical columns (object type)
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# Apply OneHotEncoding for categorical columns
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(drop='first'), cat_cols)
    ],
    remainder='passthrough'
)

X = ct.fit_transform(X)

In [5]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Models
svm = SVC()
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier()

# Train
svm.fit(X_train, y_train)
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

# Predict
svm_pred = svm.predict(X_test)
lr_pred = lr.predict(X_test)
rf_pred = rf.predict(X_test)

# Accuracy
print("Accuracy WITHOUT PCA:")
print("SVM:", accuracy_score(y_test, svm_pred))
print("Logistic Regression:", accuracy_score(y_test, lr_pred))
print("Random Forest:", accuracy_score(y_test, rf_pred))

Accuracy WITHOUT PCA:
SVM: 0.875
Logistic Regression: 0.8532608695652174
Random Forest: 0.8586956521739131


In [8]:
# Reduce dimensions (keep 95% variance)
pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X)

# Split again
X_train_pca, X_test_pca, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42
)

In [9]:
# Train again
svm.fit(X_train_pca, y_train)
lr.fit(X_train_pca, y_train)
rf.fit(X_train_pca, y_train)

# Predict
svm_pred_pca = svm.predict(X_test_pca)
lr_pred_pca = lr.predict(X_test_pca)
rf_pred_pca = rf.predict(X_test_pca)

# Accuracy
print("\nAccuracy WITH PCA:")
print("SVM:", accuracy_score(y_test, svm_pred_pca))
print("Logistic Regression:", accuracy_score(y_test, lr_pred_pca))
print("Random Forest:", accuracy_score(y_test, rf_pred_pca))


Accuracy WITH PCA:
SVM: 0.875
Logistic Regression: 0.8532608695652174
Random Forest: 0.8641304347826086
